In [1]:
import os
import requests
import pandas as pd

from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("FANTASYPROS_API_KEY")

Check API key loaded. DO NOT print it.

In [2]:
assert API_KEY is not None, "FantasyPros API key not found"
print("FantasyPros API key loaded successfully.")

FantasyPros API key loaded successfully.


Assemple API request components for player data

In [3]:
BASE_URL = "https://api.fantasypros.com/public/v2/json"

url = f"{BASE_URL}/nfl/2026/consensus-rankings"

headers = {
    "x-api-key": API_KEY
}

params = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF"
}

# Inspect, but NOT the API Key
print("URL:", url)
print("Parameters:", params)
print("API key configured:", API_KEY is not None)

URL: https://api.fantasypros.com/public/v2/json/nfl/2026/consensus-rankings
Parameters: {'position': 'ALL', 'type': 'ADP', 'scoring': 'HALF'}
API key configured: True


Make a test API call.  

NOTE - ONLY RUN THIS ONCE! Only 50 API calls/day. 

In [4]:
response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

# Inspect the player (consensus rankings) data response

In [5]:
data = response.json()

print("Status:", response.status_code)
print("Tier:", data.get("tier"))
print("Public API limited:", data.get("public_api_limited"))
print("Count:", data.get("count"))
print("Limit:", data.get("limit"))
print("Players returned:", len(data.get("players", [])))

Status: 200
Tier: premium
Public API limited: True
Count: 365
Limit: None
Players returned: 365


In [6]:
for key, value in data.items():
    if key != "players":
        print(f"{key}: {value}")

sport: NFL
type: ADP Half PPR
ranking_type_name: adp
year: 2026
week: 0
position_id: ALL
scoring: HALF
filters: 236,439,4350
count: 365
total_experts: 3
last_updated: 9/01
last_updated_ts: 1788247214
public_api_limited: True
tier: premium


In [ ]:
# # Full response data
# data

## Save player data locally

In [7]:
import json
from pathlib import Path

raw_path = Path("../data/raw/fantasypros_adp_2026_half.json")

with open(raw_path, "w") as f:
    json.dump(data, f, indent=2)

print(f"Saved raw response to: {raw_path}")

Saved raw response to: ../data/raw/fantasypros_adp_2026_half.json


Now confirm it exists & saved

In [8]:
print(raw_path.exists())
print(f"{raw_path.stat().st_size / 1024:.1f} KB")

True
312.6 KB


# Inspect Player Data

In [9]:
print(len(data["players"]))
data["players"][0]

365


{'player_id': 22968,
 'player_name': 'Jahmyr Gibbs',
 'sportsdata_id': 'fef9457e-6497-47de-9bf2-cc3b95929375',
 'player_team_id': 'DET',
 'player_position_id': 'RB',
 'player_positions': 'RB',
 'player_short_name': 'J. Gibbs',
 'player_eligibility': 'RB',
 'player_yahoo_positions': 'RB',
 'player_page_url': 'https://www.fantasypros.com/nfl/players/jahmyr-gibbs.php',
 'player_filename': 'jahmyr-gibbs.php',
 'player_yahoo_id': '40059',
 'cbs_player_id': '3162723',
 'player_bye_week': '6',
 'player_owned_avg': 99.8,
 'player_owned_espn': 99.9,
 'player_owned_yahoo': 100,
 'player_ecr_delta': None,
 'rank_ecr': 1,
 'rank_min': '1',
 'rank_max': '1',
 'rank_ave': '1.00',
 'rank_std': '0.00',
 'pos_rank': 'RB1',
 'tier': 1}

# Inspect ADP source metadata - "Experts"

ADP "Filters" from API support team:   
236 = Yahoo, 439 = RTS, 79 = ESPN, 80 = CBS, 624 = Fantrax, 4350 = Sleeper

In [12]:
experts_url = f"{BASE_URL}/nfl/2026/rankings/experts"

experts_params = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF"
}

print("URL:", experts_url)
print("Parameters:", experts_params)

URL: https://api.fantasypros.com/public/v2/json/nfl/2026/rankings/experts
Parameters: {'position': 'ALL', 'type': 'ADP', 'scoring': 'HALF'}


API CALL! - ONLY RUN ONCE!   
This is for platform-specific ADP data. 

In [13]:
experts_response = requests.get(
    experts_url,
    headers=headers,
    params=experts_params,
    timeout=30
)

print("Status code:", experts_response.status_code)

Status code: 200


In [16]:
experts_data = experts_response.json()

print(type(experts_data))
print(experts_data.keys())

<class 'dict'>
dict_keys(['sport', 'count', 'season', 'week', 'accuracy_weekly_season', 'accuracy_draft_season', 'accuracy_weekly_last_season', 'experts', 'public_api_limited', 'tier'])


In [17]:
print("Expert count:", len(experts_data["experts"]))

Expert count: 0


In [18]:
experts_data["experts"]

[]

## Retry Player Data with "Experts" 
ADP Metadata did not return "expert" data across platforms.  
236 = Yahoo, 439 = RTS, 79 = ESPN, 80 = CBS, 624 = Fantrax, 4350 = Sleeper

In [19]:
params_with_experts = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "HALF",
    "experts": "show"
}

API CALL BELOW! - Only run this once

In [20]:
response_with_experts = requests.get(
    url,                    # original consensus-rankings URL
    headers=headers,
    params=params_with_experts,
    timeout=30
)

In [21]:
data_with_experts = response_with_experts.json()

print("Status:", response_with_experts.status_code)
print("Filters:", data_with_experts.get("filters"))
print("Total experts:", data_with_experts.get("total_experts"))
print("Expert names:", data_with_experts.get("expert_name"))
print("Experts available:", data_with_experts.get("experts_available"))

Status: 200
Filters: 236,439,4350
Total experts: 3
Expert names: None
Experts available: None


In [22]:
print(data_with_experts.keys())

for key in [
    "expert_name",
    "expert_pub",
    "expert_twitter",
    "experts_available"
]:
    print(key, "->", key in data_with_experts, data_with_experts.get(key))

dict_keys(['sport', 'type', 'ranking_type_name', 'year', 'week', 'position_id', 'scoring', 'filters', 'count', 'total_experts', 'last_updated', 'players', 'last_updated_ts', 'expert_pub', 'expert_names', 'expert_twitter', 'public_api_limited', 'tier'])
expert_name -> False None
expert_pub -> True {'236': '2026-09-01 07:20:05', '439': '2026-08-31 05:30:06', '4350': '2026-09-01 07:20:14'}
expert_twitter -> True {'236': None, '439': None, '4350': 'SleeperHQ'}
experts_available -> False None


## Explore source IDs by "Expert" (platform)
Expert 4350 == Sleeper.  
236 = Yahoo, 439 = RTS, 79 = ESPN, 80 = CBS, 624 = Fantrax, 4350 = Sleeper

3 API Calls below!

In [23]:
data_with_experts["players"][0]

{'player_id': 22968,
 'player_name': 'Jahmyr Gibbs',
 'sportsdata_id': 'fef9457e-6497-47de-9bf2-cc3b95929375',
 'player_team_id': 'DET',
 'player_position_id': 'RB',
 'player_positions': 'RB',
 'player_short_name': 'J. Gibbs',
 'player_eligibility': 'RB',
 'player_yahoo_positions': 'RB',
 'player_page_url': 'https://www.fantasypros.com/nfl/players/jahmyr-gibbs.php',
 'player_filename': 'jahmyr-gibbs.php',
 'player_yahoo_id': '40059',
 'cbs_player_id': '3162723',
 'player_bye_week': '6',
 'player_owned_avg': 99.8,
 'player_owned_espn': 99.9,
 'player_owned_yahoo': 100,
 'player_ecr_delta': None,
 'rank_ecr': 1,
 'rank_min': '1',
 'rank_max': '1',
 'rank_ave': '1.00',
 'rank_std': '0.00',
 'experts': {'236': '1', '439': '1', '4350': '1'},
 'rank_points': 894,
 'pos_rank': 'RB1'}

In [24]:
puka_dict = next(
    player
    for player in data_with_experts["players"]
    if player["player_name"] == "Puka Nacua"
)

puka_dict["experts"]

{'236': '4', '439': '4', '4350': '6'}

## Test ESPN PPR ADP

FantasyPros support confirmed that ESPN ADP is available only under PPR scoring
and uses expert/source ID `79`.

Test the PPR consensus-ranking response with `experts=show` to confirm that
ESPN's individual ADP is exposed under each player's `experts` dictionary.

In [25]:
ppr_params_with_experts = {
    "position": "ALL",
    "type": "ADP",
    "scoring": "PPR",
    "experts": "show"
}

print("URL:", url)
print("Parameters:", ppr_params_with_experts)

URL: https://api.fantasypros.com/public/v2/json/nfl/2026/consensus-rankings
Parameters: {'position': 'ALL', 'type': 'ADP', 'scoring': 'PPR', 'experts': 'show'}


In [26]:
# API CALL! - Only run this once

ppr_response_with_experts = requests.get(
    url,
    headers=headers,
    params=ppr_params_with_experts,
    timeout=30
)

print("Status:", ppr_response_with_experts.status_code)

Status: 200


In [27]:
ppr_data_with_experts = ppr_response_with_experts.json()

print("Scoring:", ppr_data_with_experts.get("scoring"))
print("Filters:", ppr_data_with_experts.get("filters"))
print("Total experts:", ppr_data_with_experts.get("total_experts"))
print("Player count:", len(ppr_data_with_experts.get("players", [])))

Scoring: PPR
Filters: 79,80,439,624,4350
Total experts: 5
Player count: 699


In [28]:
# Inspect Puka Nacua
puka_ppr_dict = next(
    player
    for player in ppr_data_with_experts["players"]
    if player["player_name"] == "Puka Nacua"
)

print("Puka ECR:", puka_ppr_dict["rank_ecr"])
print("Puka experts:", puka_ppr_dict["experts"])

Puka ECR: 4
Puka experts: {'79': '4', '80': '5', '439': '4', '624': '4', '4350': '4'}


## Validate Platform-Specific ADP Coverage

Before productionizing platform-specific ADP, validate coverage across the
Half-PPR consensus-ranking player universe.

Expected sources:
- Yahoo Half-PPR: `236`
- RTSports Half-PPR: `439`
- Sleeper Half-PPR: `4350`
- ESPN PPR: `79`

The Half-PPR consensus dataset remains the primary player universe.

In [29]:
import pandas as pd

# Primary draft-board universe from the Half-PPR response.
adp_coverage_df = pd.DataFrame(
    [
        {
            "player_id": player["player_id"],
            "player_name": player["player_name"],
            "rank_ecr": player["rank_ecr"],
            "yahoo_adp_half": player.get("experts", {}).get("236"),
            "rtsports_adp_half": player.get("experts", {}).get("439"),
            "sleeper_adp_half": player.get("experts", {}).get("4350"),
        }
        for player in data_with_experts["players"]
    ]
)

adp_coverage_df.head()

,player_id,player_name,rank_ecr,yahoo_adp_half,rtsports_adp_half,sleeper_adp_half
0,22968,Jahmyr Gibbs,1,1,1,1
1,23133,Bijan Robinson,2,2,2,2
2,19788,Ja'Marr Chase,3,3,3,3
3,23180,Puka Nacua,4,4,4,6
4,16393,Christian McCaffrey,5,5,5,4


In [30]:
espn_adp_by_player_id = {
    player["player_id"]: player.get("experts", {}).get("79")
    for player in ppr_data_with_experts["players"]
}

adp_coverage_df["espn_adp_ppr"] = (
    adp_coverage_df["player_id"].map(espn_adp_by_player_id)
)

adp_coverage_df.head()

,player_id,player_name,rank_ecr,yahoo_adp_half,rtsports_adp_half,sleeper_adp_half,espn_adp_ppr
0,22968,Jahmyr Gibbs,1,1,1,1,1
1,23133,Bijan Robinson,2,2,2,2,2
2,19788,Ja'Marr Chase,3,3,3,3,3
3,23180,Puka Nacua,4,4,4,6,4
4,16393,Christian McCaffrey,5,5,5,4,7


In [31]:
# Check coverage
adp_columns = [
    "yahoo_adp_half",
    "rtsports_adp_half",
    "sleeper_adp_half",
    "espn_adp_ppr",
]

coverage_summary = pd.DataFrame(
    {
        "players_with_adp": adp_coverage_df[adp_columns].notna().sum(),
        "players_missing_adp": adp_coverage_df[adp_columns].isna().sum(),
        "coverage_pct": adp_coverage_df[adp_columns].notna().mean() * 100,
    }
)

coverage_summary.round(1)

,players_with_adp,players_missing_adp,coverage_pct
yahoo_adp_half,224,141,61.4
rtsports_adp_half,298,67,81.6
sleeper_adp_half,320,45,87.7
espn_adp_ppr,242,123,66.3


In [32]:
# Compare ADP coverage for the most draft-relevant players.

for top_n in [50, 100, 150, 200]:
    top_players_df = adp_coverage_df.nsmallest(top_n, "rank_ecr")

    print(f"\nTop {top_n} players by ECR:")

    for column in adp_columns:
        coverage_pct = top_players_df[column].notna().mean() * 100
        print(f"  {column}: {coverage_pct:.1f}%")


Top 50 players by ECR:
  yahoo_adp_half: 100.0%
  rtsports_adp_half: 100.0%
  sleeper_adp_half: 100.0%
  espn_adp_ppr: 100.0%

Top 100 players by ECR:
  yahoo_adp_half: 100.0%
  rtsports_adp_half: 100.0%
  sleeper_adp_half: 100.0%
  espn_adp_ppr: 100.0%

Top 150 players by ECR:
  yahoo_adp_half: 100.0%
  rtsports_adp_half: 100.0%
  sleeper_adp_half: 100.0%
  espn_adp_ppr: 100.0%

Top 200 players by ECR:
  yahoo_adp_half: 100.0%
  rtsports_adp_half: 100.0%
  sleeper_adp_half: 100.0%
  espn_adp_ppr: 96.5%


### Above findings were added to src by Claude. Test below:

In [33]:
from fantasy_football.extract.fantasypros import (
    fetch_consensus_adp,
    fetch_consensus_adp_ppr,
    save_raw_response,
    CONSENSUS_ADP_PPR_RAW_FILENAME,
)

# --- 1 real API call: fresh Half-PPR consensus ADP (now includes experts=show) ---
half_raw = fetch_consensus_adp()
half_path = save_raw_response(half_raw)  # -> data/raw/fantasypros_consensus_adp_2026_half.json
print("HALF cached ->", half_path)
print("HALF players:", len(half_raw.get("players", [])),
      "| sample experts:", half_raw["players"][0].get("experts"))

# --- 1 real API call: fresh PPR consensus ADP (for ESPN enrichment) ---
ppr_raw = fetch_consensus_adp_ppr()
ppr_path = save_raw_response(ppr_raw, filename=CONSENSUS_ADP_PPR_RAW_FILENAME)  # -> data/raw/fantasypros_consensus_adp_2026_ppr.json
print("PPR cached ->", ppr_path)
print("PPR players:", len(ppr_raw.get("players", [])),
      "| sample experts:", ppr_raw["players"][0].get("experts"))

HALF cached -> /Users/braddotson/Desktop/Github/Fantasy_Football_Data_Pipeline/data/raw/fantasypros_consensus_adp_2026_half.json
HALF players: 365 | sample experts: {'236': '1', '439': '1', '4350': '1'}
PPR cached -> /Users/braddotson/Desktop/Github/Fantasy_Football_Data_Pipeline/data/raw/fantasypros_consensus_adp_2026_ppr.json
PPR players: 699 | sample experts: {'79': '1', '80': '1', '439': '1', '624': '1', '4350': '1'}


# Now try "ranknings" endpoint

In [ ]:
rankings_url = f"{BASE_URL}/nfl/2026/rankings"

rankings_params = {
    "week": 0
}

print("URL:", rankings_url)
print("Parameters:", rankings_params)

API Call!

In [ ]:
rankings_response = requests.get(
    rankings_url,
    headers=headers,
    params=rankings_params,
    timeout=30
)

print("Status:", rankings_response.status_code)

### Inspect rankings response

In [ ]:
rankings_data = rankings_response.json()

print(type(rankings_data))
print(rankings_data.keys())

In [ ]:
print("experts type:", type(rankings_data["experts"]))
print("players type:", type(rankings_data["players"]))
print("ecr_experts type:", type(rankings_data["ecr_experts"]))

In [ ]:
print("experts count:", len(rankings_data["experts"]))
print("players count:", len(rankings_data["players"]))

In [ ]:
rankings_data["experts"].keys()

In [ ]:
rankings_data["players"][0]

In [ ]:
print(rankings_data["ecr_experts"].keys())

In [ ]:
print(type(rankings_data["experts"]["HALF"]))
rankings_data["experts"]["HALF"]

In [ ]:
print(type(rankings_data["ecr_experts"]["HALF"]))
rankings_data["ecr_experts"]["HALF"]

In [ ]:
gibbs = next(
    player for player in rankings_data["players"]
    if player["player_name"] == "Jahmyr Gibbs"
)

gibbs["rank"]

# Explore 2026 Preseason Projections

In [ ]:
BASE_URL = "https://api.fantasypros.com/public/v2/json"

projections_url = f"{BASE_URL}/nfl/2026/projections"

headers = {
    "x-api-key": API_KEY
}

projections_params = {
    "week": 0,
    "position": "ALL",
    "scoring": "HALF"
}

print("URL:", projections_url)
print("Parameters:", projections_params)

API CALL! - run once

In [ ]:
projections_response = requests.get(
    projections_url,
    headers=headers,
    params=projections_params,
    timeout=30
)

print("Status:", projections_response.status_code)

In [ ]:
projections_data = projections_response.json()

print("Response type:", type(projections_data))
print("Top-level keys:", projections_data.keys())

In [ ]:
# Love this cell for understanding API response structure
for key, value in projections_data.items():
    if not isinstance(value, (list, dict)):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {type(value).__name__} with {len(value)} items")

In [ ]:
projections_data["players"][0]

In [ ]:
[ # Confirms that fpid here == player_id in consensus data
    player for player in projections_data["players"]
    if player.get("name") == "Jahmyr Gibbs"
][0]

In [ ]:
# Check that ids match acrosss the whole dataset

projection_ids_set = {
    player["fpid"]
    for player in projections_data["players"]
}

player_ids_set = {
    player["player_id"]
    for player in data["players"]
}

match_count = len(player_ids_set & projection_ids_set)
total_count = len(player_ids_set)

match_count, total_count, match_count / total_count


In [ ]:
# Inspect missing players

import pandas as pd

missing_players_df = pd.DataFrame([
    player
    for player in data["players"]
    if player["player_id"] in missing_projection_ids
])

missing_players_df[
    ["player_name", "player_team_id", "player_position_id", "player_id"]
]

## Sanity check on Claude's projection data --> src code

The above was sent to Claude to productionalize in src code. Check below

In [ ]:
from fantasy_football.extract.fantasypros import (
    fetch_projections,
    save_raw_response,
    PROJECTIONS_RAW_FILENAME,
)

In [ ]:
# API CALL! - run once

projections_dict_raw = fetch_projections()

In [ ]:
print("season:", projections_dict_raw.get("season"))
print("week:", projections_dict_raw.get("week"))
print("count:", projections_dict_raw.get("count"))
print("positions:", projections_dict_raw.get("positions"))
print("tier:", projections_dict_raw.get("tier"))

print(
    "Josh Allen projected HALF:",
    projections_dict_raw["players"][0]["stats"]["points_half"]
)

In [ ]:
saved_path = save_raw_response(
    projections_dict_raw,
    filename=PROJECTIONS_RAW_FILENAME
)

print(saved_path)

# Explore 2025 Season-Long Player Points

In [ ]:
# Self-contain this ssection with imports and function definitions
import os
import requests
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ["FANTASYPROS_API_KEY"]

BASE_URL = "https://api.fantasypros.com/public/v2/json"

player_points_url = f"{BASE_URL}/nfl/2025/player-points"

headers = {
    "x-api-key": api_key
}

player_points_params = {
    "position": "ALL",
    "scoring": "HALF",
    "start": 1,
    "end": 18,
}

print("URL:", player_points_url)
print("Parameters:", player_points_params)

In [ ]:
# API CALL! - run once

player_points_response = requests.get(
    player_points_url,
    headers=headers,
    params=player_points_params,
    timeout=30
)

print("Status:", player_points_response.status_code)

In [ ]:
player_points_data = player_points_response.json()

print("Response type:", type(player_points_data))
print("Top-level keys:", player_points_data.keys())

In [ ]:
# Love this cell for understanding API response structure
for key, value in player_points_data.items():
    if not isinstance(value, (list, dict)):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {type(value).__name__} with {len(value)} items")

In [ ]:
player_points_data["players"][0]

In [ ]:
[
    player
    for player in player_points_data["players"]
    if player.get("player_id") == 22968
][0]

In [ ]:
# Load cached data for ID validation. Load locally to avoid API call
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

consensus_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "fantasypros_consensus_adp_2026_half.json"
)

with consensus_path.open("r", encoding="utf-8") as f:
    consensus_data = json.load(f)

In [ ]:
# Check player_id coverage against the 2026 draft-board universe

player_points_ids_set = {
    player["player_id"]
    for player in player_points_data["players"]
}

consensus_player_ids_set = {
    player["player_id"]
    for player in consensus_data["players"]
}

match_count = len(
    consensus_player_ids_set & player_points_ids_set
)

total_count = len(consensus_player_ids_set)

match_count, total_count, match_count / total_count

In [ ]:
import pandas as pd
# Inspect missing players
missing_2025_points_ids = (
    consensus_player_ids_set - player_points_ids_set
)

missing_2025_players_df = pd.DataFrame([
    player
    for player in consensus_data["players"]
    if player["player_id"] in missing_2025_points_ids
])

missing_2025_players_df[
    [
        "player_name",
        "player_team_id",
        "player_position_id",
        "rank_ecr",
        "pos_rank",
    ]
]

In [ ]:
missing_2025_players_df[
    ["player_name", "rank_ecr", "pos_rank", "player_team_id"]
].sort_values("rank_ecr").head(30)

In [ ]:
pd.cut(
    missing_2025_players_df["rank_ecr"],
    bins=[0, 50, 100, 150, 200, 250, 300, 350],
).value_counts().sort_index()

## Sanity check on Claude's 2025 performance data --> src code

The above was sent to Claude to productionalize in src code. Check below

In [ ]:
from fantasy_football.extract.fantasypros import (
    fetch_player_points,
    save_raw_response,
    PLAYER_POINTS_RAW_FILENAME,
)

player_points_dict_raw = fetch_player_points()

print("season:", player_points_dict_raw.get("season"))
print("scoring:", player_points_dict_raw.get("scoring"))
print("player records:", len(player_points_dict_raw.get("players", [])))

In [ ]:
# Looks good. Save to local
saved_path = save_raw_response(
    player_points_dict_raw,
    filename=PLAYER_POINTS_RAW_FILENAME,
)

print(saved_path)

# Explore 2026 Preseason Injury Data

In [ ]:
import os
import requests
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

api_key = os.environ["FANTASYPROS_API_KEY"]

BASE_URL = "https://api.fantasypros.com/public/v2/json"

injuries_url = f"{BASE_URL}/nfl/injuries"

headers = {
    "x-api-key": api_key
}

injuries_params = {
    "year": 2026,
    "week": 0,
}

print("URL:", injuries_url)
print("Parameters:", injuries_params)

In [ ]:
# API CALL! - run once

injuries_response = requests.get(
    injuries_url,
    headers=headers,
    params=injuries_params,
    timeout=30
)

print("Status:", injuries_response.status_code)

In [ ]:
injuries_data = injuries_response.json()

print("Response type:", type(injuries_data))
print("Top-level keys:", injuries_data.keys())

for key, value in injuries_data.items():
    if not isinstance(value, (list, dict)):
        print(f"{key}: {value}")
    else:
        print(f"{key}: {type(value).__name__} with {len(value)} items")

In [ ]:
injuries_data["injuries"][0]

In [ ]:
import pandas as pd

injuries_df = pd.DataFrame(injuries_data["injuries"])

injuries_df[
    [
        "player_id",
        "name",
        "team_id",
        "position_id",
        "status",
        "status_short",
        "injury_type",
        "comment",
        "injury_update_date",
        "probability_of_playing",
    ]
].sort_values("name")